In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("OlistData-New5")\
        .getOrCreate() 

In [0]:
#connect ADLSgen2 to Databricks

spark.conf.set("fs.azure.account.key.adlsgen2spark01.dfs.core.windows.net",
                 "Storageaccount -> Security + Networking -> Access Keys-->Key1valuePaste here ")
#basePath
adlsgen2CSVpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/"
adlsgen2PARQUETpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/parquet/"

#read data
customers_df = spark.read.csv(adlsgen2CSVpath + "olist_customers_dataset", header=True, inferSchema=True) 
geolocation_df = spark.read.csv(adlsgen2CSVpath + "olist_geolocation_dataset", header=True, inferSchema=True) 
order_items_df = spark.read.csv(adlsgen2CSVpath + "olist_order_items_dataset", header=True, inferSchema=True) 
payments_df = spark.read.csv(adlsgen2CSVpath + "olist_order_payments_dataset", header=True, inferSchema=True) 
reviews_df = spark.read.csv(adlsgen2CSVpath + "olist_order_reviews_dataset", header=True, inferSchema=True) 
orders_df = spark.read.csv(adlsgen2CSVpath + "olist_orders_dataset", header=True, inferSchema=True) 
products_df = spark.read.csv(adlsgen2CSVpath + "olist_products_dataset", header=True, inferSchema=True) 
sellers_df = spark.read.csv(adlsgen2CSVpath + "olist_sellers_dataset", header=True, inferSchema=True) 
catgeory_translation_df = spark.read.csv(adlsgen2CSVpath + "product_category_name_translation", header=True, inferSchema=True) 

In [0]:
#Cache feqquently used data for better performance

orders_df.cache()
customers_df.cache()
order_items_df.cache()


DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [0]:
#lets say wanted ifnormation about ORDERS so staretd inner join with orders - lets say if custoemr was the main goal then start with that

orders_items_joined_df = orders_df.join(order_items_df,'order_id','inner')

In [0]:
orders_items_products_df = orders_items_joined_df.join(products_df,'product_id','inner')

In [0]:
orders_items_products_sellers_df = orders_items_products_df.join(sellers_df,'seller_id','inner')

In [0]:
full_orders_df = orders_items_products_sellers_df.join(customers_df,'customer_id','inner')

In [0]:
#GeoLocation Data
#so just enriching the Geolocation data, to my main orders DATA  , as we dont want to delete teh order if geolocation is nULL -- so taking LEFT join 
full_orders_df = full_orders_df.join(geolocation_df,full_orders_df['customer_zip_code_prefix'] == geolocation_df['geolocation_zip_code_prefix'],'left')


In [0]:
geolocation_df.printSchema(), full_orders_df.printSchema()

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable

(None, None)

In [0]:
#Reviews
#same if reviews are there then take thsoe also - so Left join

full_orders_df = full_orders_df.join(reviews_df,'order_id','left')

In [0]:
#Payments
full_orders_df = full_orders_df.join(payments_df,'order_id','left')

In [0]:
#cache as will be quwerying it frequently
full_orders_df.cache()

DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [0]:
from pyspark.sql.functions import *

In [0]:
#Total revenues per seller

seller_revenue_df = full_orders_df.groupBy('seller_id').agg(sum('price').alias('total_revenue'))

In [0]:
#NOW this is the 1st ACtion that we are doing afetr lot of transofrmations , so spark will take time
#1. do internal optimization, will run all above, as LAZY EVALUATION -- so thats benefit , when now i acttualy need it so all is getting evaluated

seller_revenue_df.show()

+--------------------+--------------------+
|           seller_id|       total_revenue|
+--------------------+--------------------+
|8e6cc767478edae94...|  1145757.4000000027|
|ff063b022a9a0aab9...|           1860394.0|
|0ea22c1cfbdc755f8...|  1747345.2000000058|
|6bcb2352a2a0c38b8...|  2830.2000000000025|
|ef0ace09169ac0905...|  2261367.7000000034|
|f457c46070d02cadd...|  3812449.4999999925|
|537eb890efff034a8...|   894311.9000000029|
|e3dd723429d1b2361...|   66929.36999999998|
|31561f325664a8a7a...|   297226.7700000004|
|bbf9ad41dca6603e6...|           1083065.0|
|9c0e69c7bf2619675...|   5791323.660000035|
|a673821011d0cec28...|  2175949.9899999993|
|a420f60ff1aa9acc8...|            999172.5|
|b76dba6c951ab00dc...|   302582.6599999984|
|900ba814c251a6925...|   4572013.930000009|
|fad44952713764836...|            934189.0|
|7a67c85e85bb2ce85...|2.0312794889999893E7|
|d51e0a403fe2e689e...|           1276985.0|
|9d213f303afae4983...|   2321.400000000004|
|5343d0649eca2a983...|  1486447.

In [0]:
#Total orders per customer
#Average Review score per seller
#Most SOld products (Top 10)
# Top customers by spending


In [0]:
customer_total_orders = full_orders_df.groupBy('customer_id').agg(count('order_id').alias('total_order_per_customer')).orderBy(col('total_order_per_customer').desc())

In [0]:
customer_total_orders.show()

+--------------------+------------------------+
|         customer_id|total_order_per_customer|
+--------------------+------------------------+
|351e40989da90e704...|                   11427|
|50920f8cd0681fd86...|                   10752|
|9b43e2a62de9bab3a...|                    8556|
|270c23a11d024a44c...|                    8001|
|d3e82ccec3cb5f956...|                    6876|
|5c87184371002d49e...|                    6876|
|d5f2b3f597c7ccafb...|                    6706|
|c2f18647725395af4...|                    6612|
|24e7dc2ff8c071263...|                    6597|
|7bb57d182bdc11653...|                    6258|
|d22f25a9fadfb1abb...|                    6072|
|63b964e79dee32a35...|                    6072|
|1ff773612ab8934db...|                    5820|
|13aa59158da63ba0e...|                    5206|
|78fc46047c4a639e8...|                    5200|
|dd3f1762eb601f41c...|                    4992|
|a193aa8d905b8e246...|                    4896|
|9eb3d566e87289dcb...|                  

In [0]:
seller_avg_review_df = full_orders_df.groupBy('seller_id').agg(avg('review_score').alias('average_review'))

In [0]:
seller_avg_review_df.show()

+--------------------+------------------+
|           seller_id|    average_review|
+--------------------+------------------+
|8e6cc767478edae94...|  3.89280737485261|
|ff063b022a9a0aab9...|3.9920140028443276|
|0ea22c1cfbdc755f8...| 4.231254932912392|
|6bcb2352a2a0c38b8...|2.4831460674157304|
|ef0ace09169ac0905...|  4.32053782202607|
|f457c46070d02cadd...|3.5567388268156424|
|537eb890efff034a8...| 4.103664313437854|
|e3dd723429d1b2361...|  3.34984984984985|
|31561f325664a8a7a...| 4.620727432077126|
|bbf9ad41dca6603e6...|  4.87277628032345|
|9c0e69c7bf2619675...|3.7712917879389605|
|a673821011d0cec28...|  4.24530490252933|
|a420f60ff1aa9acc8...|3.4408756333046555|
|b76dba6c951ab00dc...| 4.219273172723561|
|900ba814c251a6925...| 4.195056250990334|
|fad44952713764836...|            4.3125|
|7a67c85e85bb2ce85...| 4.258920734844587|
|d51e0a403fe2e689e...| 4.001885458402074|
|9d213f303afae4983...|               5.0|
|5343d0649eca2a983...| 3.415879884605298|
+--------------------+------------

In [0]:
most_sold_products_df = full_orders_df.groupBy('product_id').agg(count('order_id').alias('total_sold')).orderBy(col('total_sold').desc()).limit(10)

In [0]:
most_sold_products_df.show()

+--------------------+----------+
|          product_id|total_sold|
+--------------------+----------+
|aca2eb7d00ea1a7b8...|     86740|
|422879e10f4668299...|     81110|
|99a4788cb24856965...|     78775|
|389d119b48cf3043d...|     60248|
|d1c427060a0f73f6b...|     59274|
|368c6c730842d7801...|     58358|
|53759a2ecddad2bb8...|     52654|
|53b36df67ebb7c415...|     52105|
|154e7e31ebfa09220...|     42700|
|3dd2a17168ec895c7...|     40787|
+--------------------+----------+



In [0]:
top_customers_df = full_orders_df.groupBy('customer_id').agg(sum('price').alias('total_spending')).orderBy(col('total_spending').desc()).limit(10)
display(top_customers_df)

customer_id,total_spending
d3e82ccec3cb5f956a38d96c057ceaae,6662844.0
df55c14d1476a9a3467f131269c2477f,3565657.0
fe5113a38e3575c04f5a3413100d4e48,3293604.0
ec5b2ba62e574342386871631fafd3fc,2556120.0
63b964e79dee32a3587651701a2b8dbf,2501664.0
46bb3c0b1a65c8399d0363cefbcc4f37,2336752.0
05455dfa7cd02f13d132aa7a6a9729c6,2160194.400000087
3690e975641f01bd07ad635f03dbb894,2124498.0
349509b216bd5ec11c5fae929fd13595,1923627.0
695476b5848d64ba0875324c88390206,1820543.1299999943


# Optimized Joins for Data Integration

1. lets say we identified that sellers_df, geolocation_df , reviews_df has smaller data
2. so we can BROADCAST them while joining to optimize teh performance , while running jobs -- liek while analyzing liek above Total evenue and all to find will become faster


In [0]:
from pyspark.sql.functions import *

In [0]:
orders_items_joined_df = orders_df.join(order_items_df,'order_id','inner')

In [0]:
orders_items_products_df = orders_items_joined_df.join(products_df,'product_id','inner')

In [0]:
orders_items_products_sellers_df = orders_items_products_df.join(broadcast(sellers_df),'seller_id','inner')

In [0]:
full_orders_df = orders_items_products_sellers_df.join(customers_df,'customer_id','inner')

In [0]:
full_orders_df = full_orders_df.join(broadcast(geolocation_df),full_orders_df['customer_zip_code_prefix'] == geolocation_df['geolocation_zip_code_prefix'],'left')

In [0]:
full_orders_df = full_orders_df.join(broadcast(reviews_df),'order_id','left')

In [0]:
full_orders_df = full_orders_df.join(payments_df,'order_id','left')

In [0]:
full_orders_df.cache()

DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

#Aggregation

In [0]:
#do above transfoprmations here, again on the optimized join now -

#Window Function and Ranking

In [0]:
from pyspark.sql.window import Window

In [0]:
window_spec = Window.partitionBy('seller_id').orderBy(desc('price'))

In [0]:
#Rank Top Selling Products Per Seller

top_selling_products_per_seller_df = full_orders_df.withColumn('rank',rank().over(window_spec)).filter(col('rank')<=5).select('seller_id','product_id','rank','price')

In [0]:
top_selling_products_per_seller_df.show()

+--------------------+--------------------+----+-----+
|           seller_id|          product_id|rank|price|
+--------------------+--------------------+----+-----+
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2db000af6...|a2ff5a97bf95719e3...|   1|895.0|
|0015a82c2

#Dense Rank for Sellers  Based on Revenue
1. Granuality - find total revenue 1st 

In [0]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [0]:

seller_total_revenue_df = full_orders_df.groupBy('seller_id').agg(sum('price').alias('total_revenue')).select('seller_id','total_revenue')

In [0]:
seller_total_revenue_df.show()

+--------------------+------------------+
|           seller_id|     total_revenue|
+--------------------+------------------+
|8e6cc767478edae94...|1145757.4000000046|
|6eeed17989b0ae47c...|           20068.0|
|0ea22c1cfbdc755f8...|1747345.2000002754|
|062ce95fa2ad4dfae...|1120347.8000000264|
|a49928bcdf77c55c6...|1220624.6000000024|
|e63e8bfa530fb1691...| 219480.9999999959|
|9c068d10aca38e85c...|311486.79999999877|
|ff063b022a9a0aab9...|         1860394.0|
|ec8879960bd2221d5...| 732805.3999999815|
|da7039f29f90ce5b4...|131397.30000000095|
|9803a40e82e45418a...|          387371.5|
|c522be04e020c1e7b...| 76932.80000000035|
|2009a095de2a2a416...| 49265.33000000031|
|a3082f442524a1be4...|   70512.300000001|
|4d600e08ecbe08258...|436434.23000000074|
|b3f19518fcec265b2...| 66204.66999999991|
|0b64bcdb0784abc13...| 40003.89999999974|
|791cfcfe22fe4a771...|176194.83000000136|
|7aa4334be125fcdd2...| 2509294.490000046|
|9c690ceacd5c66731...|114021.30000000098|
+--------------------+------------

In [0]:
window_spec2 = Window.orderBy(desc('total_revenue'))

In [0]:

top_sellers_df = seller_total_revenue_df.withColumn('denseRank',dense_rank().over(window_spec2)).filter(col('denseRank') == 2).select("seller_id","total_revenue","denseRank").orderBy(col('denseRank').desc())

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
top_sellers_df.show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------------+-------------------+---------+
|           seller_id|      total_revenue|denseRank|
+--------------------+-------------------+---------+
|53243585a1d6dc264...|3.429159295000153E7|        2|
+--------------------+-------------------+---------+



# Advance Aggregation and Enrichment

In [0]:
#Total Revenue &Average Order value(AOV) per customer
#MAU - Montly active users
#DAU - Daily active users

In [0]:
customer_spending_df = full_orders_df.groupBy('customer_id')\
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_spent'),
        round(avg('price'),2).alias('AOV')
    )\
        .orderBy(desc('total_spent'))

customer_spending_df.show()

+--------------------+------------+------------------+-------+
|         customer_id|total_orders|       total_spent|    AOV|
+--------------------+------------+------------------+-------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|
|695476b5848d64ba0...|         687|1820543.1299999943|2649.99|
|73236a0796f53d60d...|         832|         1755520.0| 2110.0|
|cc803a2c412833101...|         762|         1676400.0| 2200.0|
|1ff773612ab8934db...|        5820|1658641.7999999512| 

In [0]:
6662844.0/6876

969.0

In [0]:
#Seller Performance metrics (Revenue, Average Revuew, Order Count)

seller_performance_df = full_orders_df.groupBy('seller_id')\
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_revenue'),
        round(avg('review_score'),2).alias('avg_review_score'),
        round(stddev('price'),2).alias('price_variability')

    )\
        .orderBy(desc('total_revenue'))

In [0]:
seller_performance_df.show()

+--------------------+------------+--------------------+----------------+-----------------+
|           seller_id|total_orders|       total_revenue|avg_review_score|price_variability|
+--------------------+------------+--------------------+----------------+-----------------+
|4869f7a5dfa277a7d...|      184587|3.6138717320015594E7|            4.09|           111.65|
|53243585a1d6dc264...|       54514| 3.429159295000153E7|            4.12|           499.65|
|4a3ca9315b744ce9f...|      330661|  3.37595708400352E7|            3.77|            59.37|
|7c67e1448b00f6e96...|      233306|3.2282321790031552E7|            3.42|            50.39|
|fa1c13f2614d7b5c4...|       87686|3.0139386310007345E7|            4.38|            307.7|
|da8622b14eb17ae28...|      264433|2.9857669730032437E7|            3.98|            72.92|
|7e93a43ef30c4f03f...|       50226|2.6315706300003525E7|            4.15|           377.24|
|1025f0e2d44d7041d...|      229587|2.2937518520002756E7|            3.89|       

In [0]:
#Product popularity metrics

product_metrics_df = full_orders_df.groupBy('product_id')\
    .agg(
        count('order_id').alias('total_sales'),
        sum('price').alias('total_revenue'),
        round(avg('price'),2).alias('avg_price'),
        round(stddev('price'),2).alias('price_volatility'),
        collect_set('seller_id').alias('unique_sellers')
    )\
        .orderBy(desc('total_sales'))

In [0]:
product_metrics_df.show()

+--------------------+-----------+------------------+---------+----------------+--------------------+
|          product_id|total_sales|     total_revenue|avg_price|price_volatility|      unique_sellers|
+--------------------+-----------+------------------+---------+----------------+--------------------+
|aca2eb7d00ea1a7b8...|      86740| 6164630.299998479|    71.07|            3.17|[955fee9216a65b61...|
|422879e10f4668299...|      81110| 4442791.510000359|    54.77|            4.46|[1f50f920176fa81d...|
|99a4788cb24856965...|      78775|  6921762.70999809|    87.87|            4.08|[53d00c40e32aeb92...|
|389d119b48cf3043d...|      60248| 3280533.130000726|    54.45|            4.37|[1f50f920176fa81d...|
|d1c427060a0f73f6b...|      59274| 8220103.330000201|   138.68|           16.58|[a1043bafd471dff5...|
|368c6c730842d7801...|      58358|3181698.9000006923|    54.52|            4.59|[1f50f920176fa81d...|
|53759a2ecddad2bb8...|      52654|2893017.5000006156|    54.94|            4.52|[1

#
Monthly Revenue and Order Count Trend

total_orders
total_revenue
avg_order_value
min_order_value
max_order_value

In [0]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [0]:


full_orders_month_df = full_orders_df.withColumn('order_purchase_month',month(col("order_purchase_timestamp")))



In [0]:
full_orders_month_df.select('customer_id','order_purchase_timestamp','order_purchase_month').show()

+--------------------+------------------------+--------------------+
|         customer_id|order_purchase_timestamp|order_purchase_month|
+--------------------+------------------------+--------------------+
|9c6ea4e6fbcff50e7...|     2018-03-01 17:10:26|                   3|
|2ea5497e8e41a40f8...|     2018-07-04 22:44:05|                   7|
|74181092ec10162ba...|     2017-06-28 16:48:44|                   6|
|905d0f8557f4baccd...|     2018-07-29 15:49:54|                   7|
|7ee968b8ed74301e0...|     2018-03-19 21:50:14|                   3|
|5270e49915940227a...|     2018-08-14 09:15:11|                   8|
|f06a932d9fefe3431...|     2018-05-15 20:55:04|                   5|
|d19813c8754cc82d0...|     2018-04-03 20:44:15|                   4|
|746704bb5f6ca6291...|     2017-09-28 13:32:31|                   9|
|f34cd6c3835485798...|     2017-11-20 22:13:47|                  11|
|febea3481938a4b17...|     2017-12-07 08:16:34|                  12|
|9f5aeea297d6ddacd...|     2018-01

In [0]:
#Monthly Revenue and Order Count Trend
#total_orders total_revenue avg_order_value min_order_value max_order_value

monthly_trend_df =full_orders_month_df.groupBy('order_purchase_month')\
    .agg(
        count('order_id').alias('total_order'),
        sum('price').alias('total_revenue'),
        round(avg('price'),2).alias('avg_order_value'),
        min('price').alias('min_order_value'),
        max('price').alias('max_order_value')
    )\
        .orderBy('order_purchase_month')

In [0]:
monthly_trend_df.show()

+--------------------+-----------+--------------------+---------------+---------------+---------------+
|order_purchase_month|total_order|       total_revenue|avg_order_value|min_order_value|max_order_value|
+--------------------+-----------+--------------------+---------------+---------------+---------------+
|                   1|    1495580| 1.715329014995117E8|         114.69|            2.9|         3690.0|
|                   2|    1551163|1.7878178406952173E8|         115.26|           2.99|         6735.0|
|                   3|    1809467|2.1868116842945385E8|         120.85|            4.9|        4099.99|
|                   4|    1693860| 2.171569691294695E8|          128.2|           0.85|         4799.0|
|                   5|    1918571| 2.400611519694865E8|         125.12|            3.5|         6499.0|
|                   6|    1701909|2.1024332348955354E8|         123.53|           3.49|         4590.0|
|                   7|    1847639|2.2290885709948018E8|         

In [0]:
#Customer Retention Analysis (First & Last Order)

customer_retention_df = full_orders_df.groupBy('customer_id')\
    .agg(
        min('order_purchase_timestamp').alias('first_order'),
        max('order_purchase_timestamp').alias('last_order'),
        countDistinct('order_id').alias('total_orders'),
        round(avg('price'),2).alias('aov')
    )\
        .orderBy(desc('total_orders'))

#do count Distinct = orderid - as rows can be duplicated ,due to join
#AS all CUSTOMERS have placed only one order , so in output seeing the FirstOrderdate & lastorderDate same
        
customer_retention_df_2 = full_orders_df.groupBy('customer_id')\
    .agg(
        first('order_purchase_timestamp').alias('first_order'),
        last('order_purchase_timestamp').alias('last_order'),
        count('order_id').alias('total_orders'),
        round(avg('price'),2).alias('aov')
    )\
        .orderBy(desc('total_orders'))


In [0]:
customer_retention_df.display()

customer_id,first_order,last_order,total_orders,aov
c7d5c3347080c1ba85f7c9887d8e3440,2017-12-01T10:20:48Z,2017-12-01T10:20:48Z,1,639.99
e7da42975653f8bcc8311bdef3847164,2017-12-22T11:42:09Z,2017-12-22T11:42:09Z,1,179.9
879ba647fb33e5abbc085e25df942716,2017-03-12T15:19:12Z,2017-03-12T15:19:12Z,1,155.0
c0e828e3a7e9898af73a83fe61d40c8a,2018-03-20T23:07:30Z,2018-03-20T23:07:30Z,1,129.9
b02bb20c4a867ed23194f205619ada9e,2017-08-01T13:43:33Z,2017-08-01T13:43:33Z,1,32.0
46c711a52bd83fdad273158043470bed,2018-08-09T11:36:01Z,2018-08-09T11:36:01Z,1,14.9
7004621c5c13e35086dc0e1f0a4bb19f,2017-08-22T09:49:06Z,2017-08-22T09:49:06Z,1,113.0
5affcc0cbdac28f6974a3b119801ef32,2017-12-03T19:24:22Z,2017-12-03T19:24:22Z,1,27.99
9ea39e2ec7b823ba5deb2c903ecd6d1e,2018-06-29T09:45:06Z,2018-06-29T09:45:06Z,1,69.9
49408412569c17ef3b50f801de020c3c,2018-04-13T11:40:42Z,2018-04-13T11:40:42Z,1,79.99


In [0]:
customer_retention_df_2.show()

+--------------------+-------------------+-------------------+------------+------+
|         customer_id|        first_order|         last_order|total_orders|   aov|
+--------------------+-------------------+-------------------+------------+------+
|351e40989da90e704...|2017-07-13 10:42:37|2017-07-13 10:42:37|       11427| 85.99|
|50920f8cd0681fd86...|2018-01-27 11:28:32|2018-01-27 11:28:32|       10752| 43.82|
|9b43e2a62de9bab3a...|2017-05-25 22:27:50|2017-05-25 22:27:50|        8556|  26.4|
|270c23a11d024a44c...|2017-08-08 20:26:31|2017-08-08 20:26:31|        8001| 36.59|
|d3e82ccec3cb5f956...|2017-03-18 14:28:34|2017-03-18 14:28:34|        6876| 969.0|
|5c87184371002d49e...|2018-01-05 19:15:37|2018-01-05 19:15:37|        6876| 12.49|
|d5f2b3f597c7ccafb...|2017-12-13 14:21:15|2017-12-13 14:21:15|        6706|  59.0|
|c2f18647725395af4...|2018-03-06 19:21:47|2018-03-06 19:21:47|        6612|  34.9|
|24e7dc2ff8c071263...|2017-11-24 16:16:45|2017-11-24 16:16:45|        6597|  59.2|
|7bb

In [0]:
test_df = full_orders_df.filter(col("customer_id") == '351e40989da90e70487765f6ea15d54b').select("customer_id","order_purchase_timestamp")
test_df.show()

+--------------------+------------------------+
|         customer_id|order_purchase_timestamp|
+--------------------+------------------------+
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10:42:37|
|351e40989da90e704...|     2017-07-13 10

In [0]:
orders_df.groupBy("customer_id") \
    .agg(countDistinct("order_id").alias("total_orders")) \
    .orderBy(desc("total_orders")) \
    .show(20)

    ##AS all CUSTOMERS have placed only one order , so in output seeing the FirstOrderdate & lastorderDate same

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|3f0f45bd49f790854...|           1|
|62ce8efec406efb16...|           1|
|04cef6b920c0d8f16...|           1|
|6b75e2df886b0fdde...|           1|
|088935be5a63fd416...|           1|
|fdd48a32f403170cf...|           1|
|b2e01250b17d8fca9...|           1|
|dca00fb1b6171b713...|           1|
|e87f6224556205ac0...|           1|
|ed0af0bffd15b2569...|           1|
|10242429481cedab7...|           1|
|40e7f3dd68599eba4...|           1|
|74f779df828db21f3...|           1|
|0d878c03ad06e3772...|           1|
|0632cb63610a5d7d5...|           1|
|a3f3f06060fe4f8e8...|           1|
|732594f0f8f10ebee...|           1|
|c79c8d356d92bb1b7...|           1|
|2374bb1234352ce58...|           1|
|fab1a643ad6f3da34...|           1|
+--------------------+------------+
only showing top 20 rows


#Extended Enrichment

In [0]:
#Order status Flags

full_orders_df.select('order_status').show()

+------------+
|order_status|
+------------+
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|    invoiced|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
+------------+
only showing top 20 rows


In [0]:
full_orders_df = full_orders_df.withColumn('is_delivered',when(col('order_status')=='delivered',lit(1)).otherwise(lit(0)))\
    .withColumn('is_canceled',when(col('order_status')=='canceled',lit(1)).otherwise(lit(0)))

In [0]:
full_orders_df.where(full_orders_df['order_status']=='canceled').select('order_status','is_delivered','is_canceled').show()

+------------+------------+-----------+
|order_status|is_delivered|is_canceled|
+------------+------------+-----------+
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
+------------+------------+-----------+
only showing top 20 rows


In [0]:
full_orders_df.columns

['order_id',
 'customer_id',
 'seller_id',
 'product_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_item_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state',
 'review_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'is_delivered',
 'is_canceled

In [0]:
#Order Revenue Calculation

full_orders_df = full_orders_df.withColumn('order_revenue',col('price')+col("freight_value"))

In [0]:
full_orders_df.select('price','freight_value','order_revenue').orderBy('price').display()

price,freight_value,order_revenue
0.85,22.3,23.150000000000002
0.85,18.23,19.080000000000002
0.85,22.3,23.150000000000002
0.85,18.23,19.080000000000002
0.85,22.3,23.150000000000002
0.85,18.23,19.080000000000002
0.85,18.23,19.080000000000002
0.85,22.3,23.150000000000002
0.85,18.23,19.080000000000002
0.85,18.23,19.080000000000002


#Customer Segmentation based on spending 
as premium customer- will be getting diffeent offers and all 

In [0]:


customer_spending_df = customer_spending_df.withColumn('customer_segment',
                                                       when(col('AOV') >=1200,"High-Value")
                                                       .when((col("AOV") < 1200) & (col("AOV") >= 500), "Medium-Value")
                                                       .otherwise('Low-Value') )

In [0]:
customer_spending_df.show()

+--------------------+------------+------------------+-------+----------------+
|         customer_id|total_orders|       total_spent|    AOV|customer_segment|
+--------------------+------------+------------------+-------+----------------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|    Medium-Value|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|      High-Value|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|      High-Value|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|      High-Value|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|       Low-Value|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|      High-Value|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|    Medium-Value|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|      High-Value|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|      High-Value|
|695476b5848d64ba0...|         687|18205

In [0]:
#Now joining with out master full_orders_df

In [0]:
full_orders_df = full_orders_df.join(customer_spending_df.select('customer_id','customer_segment'),on='customer_id',how='left')

In [0]:
full_orders_df.columns

['customer_id',
 'order_id',
 'seller_id',
 'product_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_item_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state',
 'review_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'is_delivered',
 'is_canceled

In [0]:
full_orders_df.select('customer_id','customer_segment').show()

+--------------------+----------------+
|         customer_id|customer_segment|
+--------------------+----------------+
|f06a932d9fefe3431...|       Low-Value|
|327b7945357e53c8e...|       Low-Value|
|252958ad9cd878e5f...|       Low-Value|
|7ee968b8ed74301e0...|       Low-Value|
|5cdc045a9409dbfa5...|       Low-Value|
|7071c00358d779ddd...|       Low-Value|
|746704bb5f6ca6291...|       Low-Value|
|38f15cd67a5309b8e...|       Low-Value|
|74181092ec10162ba...|       Low-Value|
|febea3481938a4b17...|       Low-Value|
|9c6ea4e6fbcff50e7...|       Low-Value|
|8e73500c17baa8ea7...|       Low-Value|
|138a01aa92239376f...|       Low-Value|
|f34cd6c3835485798...|       Low-Value|
|d19813c8754cc82d0...|       Low-Value|
|5270e49915940227a...|       Low-Value|
|8c36552c8308b0ba8...|       Low-Value|
|9f5aeea297d6ddacd...|       Low-Value|
|905d0f8557f4baccd...|       Low-Value|
|2ea5497e8e41a40f8...|       Low-Value|
+--------------------+----------------+
only showing top 20 rows


#Hourly order Distribution

In [0]:

full_orders_df = full_orders_df.withColumn('hour_of_day',expr('hour(order_purchase_timestamp)'))


In [0]:
full_orders_df.select('order_purchase_timestamp','hour_of_day').display()

order_purchase_timestamp,hour_of_day
2018-03-01T17:10:26Z,17
2018-07-04T22:44:05Z,22
2017-06-28T16:48:44Z,16
2018-07-29T15:49:54Z,15
2018-03-19T21:50:14Z,21
2018-08-14T09:15:11Z,9
2018-05-15T20:55:04Z,20
2018-04-03T20:44:15Z,20
2017-09-28T13:32:31Z,13
2017-11-20T22:13:47Z,22


In [0]:
#Weekday Vs Weekend Order

# used pyspark = dayofweek -- function
full_orders_df = full_orders_df.withColumn('order_day_type', when(expr('dayofweek(order_purchase_timestamp) IN (1,7)'), lit('Weekend')).otherwise(lit('Weekday')))

In [0]:
full_orders_df.select('order_purchase_timestamp','order_day_type').show()

+------------------------+--------------+
|order_purchase_timestamp|order_day_type|
+------------------------+--------------+
|     2018-03-01 17:10:26|       Weekday|
|     2018-07-04 22:44:05|       Weekday|
|     2017-06-28 16:48:44|       Weekday|
|     2018-07-29 15:49:54|       Weekend|
|     2018-03-19 21:50:14|       Weekday|
|     2018-08-14 09:15:11|       Weekday|
|     2018-05-15 20:55:04|       Weekday|
|     2018-04-03 20:44:15|       Weekday|
|     2017-09-28 13:32:31|       Weekday|
|     2017-11-20 22:13:47|       Weekday|
|     2017-12-07 08:16:34|       Weekday|
|     2018-01-26 14:47:07|       Weekday|
|     2018-04-26 12:44:27|       Weekday|
|     2017-12-18 15:09:06|       Weekday|
|     2018-07-16 16:25:10|       Weekday|
|     2018-06-11 19:41:42|       Weekday|
|     2017-04-20 16:37:54|       Weekday|
|     2018-07-31 11:14:07|       Weekday|
|     2018-03-15 14:44:52|       Weekday|
|     2018-08-14 10:47:21|       Weekday|
+------------------------+--------

# a new column frieght category  based on frieght_value --> low, med or high

In [0]:
full_orders_df.agg(max('freight_value'),min('freight_value')).show()

+------------------+------------------+
|max(freight_value)|min(freight_value)|
+------------------+------------------+
|            409.68|               0.0|
+------------------+------------------+



In [0]:


full_orders_df = full_orders_df.withColumn('freight_category', when(col("freight_value") < 100, lit('low')).when((col("freight_value") >= 100) & (col("freight_value") <= 300), lit('medium')).otherwise(lit('high')))
full_orders_df.select('freight_value','freight_category').display()

freight_value,freight_category
29.02,low
13.86,low
16.6,low
18.29,low
7.39,low
12.53,low
7.93,low
7.39,low
15.1,low
13.72,low


In [0]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [0]:
#Order Volume by Customer State

full_orders_df.groupBy('customer_state').agg(count('order_id').alias('order_volume')).orderBy(col('order_volume').desc()).show()

+--------------+------------+
|customer_state|order_volume|
+--------------+------------+
|            SP|     6742255|
|            RJ|     3626875|
|            MG|     3433239|
|            RS|      971696|
|            PR|      746540|
|            SC|      644930|
|            BA|      443992|
|            ES|      367217|
|            GO|      162430|
|            MT|      155233|
|            PE|      132005|
|            DF|      109466|
|            PA|       96279|
|            CE|       74749|
|            MS|       73693|
|            MA|       61710|
|            AL|       37742|
|            PB|       33381|
|            SE|       28146|
|            PI|       27696|
+--------------+------------+
only showing top 20 rows


#Putting the data in GOLD layer- as aggregated 

In [0]:
adlsgen2_gold_parquet = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/gold/"
full_orders_df.write.mode('overwrite').parquet(
    adlsgen2_gold_parquet + 'full_orders_df.parquet'
)